Set varibles

In [ ]:
from config import init_env
from config import variables
import importlib
variables = importlib.reload(variables)
init_env.set_environment_variables()


In [22]:
from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.models.document_grounding import (GroundingModule, GroundingType, DataRepositoryType, GroundingFilterSearch, DocumentGrounding,DocumentGroundingFilter)
from gen_ai_hub.orchestration.models.llm import LLM

## Optioal - Check System readiness
 

### Check resource group  
 

document grounding is done in a dedicated resource group "DocumentGrounding"

In [3]:
import os
os.environ['AICORE_RESOURCE_GROUP'] = 'DocumentGrounding'

In [8]:
# Create Connection
from ai_core_sdk.ai_core_v2_client import AICoreV2Client
ai_core_client = AICoreV2Client(
    base_url = os.environ['AICORE_BASE_URL'] + "/v2", # The present SAP AI Core API version is 2
    auth_url=  os.environ['AICORE_AUTH_URL'], 
    client_id = os.environ['AICORE_CLIENT_ID'],
    resource_group =os.environ['AICORE_RESOURCE_GROUP'],
    client_secret = os.environ['AICORE_CLIENT_SECRET']
)
response = ai_core_client.resource_groups.query()
print(response)

Resources: [{Resource group id: default}, {Resource group id: DocumentGrounding}, {Resource group id: himanshuksah}], Count: 3


List configurations in the resouce group

In [9]:

response = ai_core_client.configuration.query()

# Filter items with scenario_id = 'orchestration'
orchestration_items = [vars(rg) for rg in response.resources if getattr(rg, 'scenario_id', None) == 'orchestration']

# Print details of all matching items
for item in orchestration_items:
    print("--------------- Configuration ----------------")
    for key, value in item.items():
        print(f"{key}: {value}")


--------------- Configuration ----------------
id: 3a7056f3-979e-4ab8-a3db-d58a1cbcee59
name: ail-auto-orchestration
scenario_id: orchestration
executable_id: orchestration
parameter_bindings: []
input_artifact_bindings: []
created_at: 2025-09-15 10:50:10+00:00
scenario: None


List orchestration deployment using the configuration

In [10]:
response = ai_core_client.deployment.query()

# Filter items with configuration_id  
orchestration_items = [vars(rg) for rg in response.resources if getattr(rg, 'configuration_id', None) == '3a7056f3-979e-4ab8-a3db-d58a1cbcee59']

# Print details of all matching items
for item in orchestration_items:
    print("--------------- Deployment ----------------")
    for key, value in item.items():
        print(f"{key}: {value}")

--------------- Deployment ----------------
id: dcdc91cb8f73b02d
configuration_id: 3a7056f3-979e-4ab8-a3db-d58a1cbcee59
configuration_name: ail-auto-orchestration
scenario_id: orchestration
status: Status.RUNNING
target_status: TargetStatus.RUNNING
created_at: 2025-09-15 10:50:12+00:00
modified_at: 2025-11-21 01:11:47+00:00
status_message: None
status_details: None
submission_time: 2025-09-15 10:54:30+00:00
start_time: 2025-09-15 10:55:32+00:00
completion_time: None
deployment_url: https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/dcdc91cb8f73b02d
last_operation: Operation.CREATE
latest_running_configuration_id: 3a7056f3-979e-4ab8-a3db-d58a1cbcee59
details: {'scaling': {'backend_details': {}}, 'resources': {'backend_details': {}}}
ttl: None


### Check embedding model

The embedding_model is deployed in the default resouce group

In [2]:
import os
os.environ['AICORE_RESOURCE_GROUP'] = 'default'

In [76]:
# Create Connection
from ai_core_sdk.ai_core_v2_client import AICoreV2Client
ai_core_client = AICoreV2Client(
    base_url = os.environ['AICORE_BASE_URL'] + "/v2", # The present SAP AI Core API version is 2
    auth_url=  os.environ['AICORE_AUTH_URL'], 
    client_id = os.environ['AICORE_CLIENT_ID'],
    resource_group =os.environ['AICORE_RESOURCE_GROUP'],
    client_secret = os.environ['AICORE_CLIENT_SECRET']
)
response = ai_core_client.resource_groups.query()
print(response)

Resources: [{Resource group id: default}, {Resource group id: DocumentGrounding}, {Resource group id: himanshuksah}], Count: 3


To find the configuration id of the embedding model(text-embedding-3-large) from the search result

In [77]:
response = ai_core_client.configuration.query()

# Filter items with scenario_id = 'foundation-models'
foundation_model_items = [vars(rg) for rg in response.resources if getattr(rg, 'scenario_id', None) == 'foundation-models']

# Print details of all matching items
for item in foundation_model_items:
    print("--------------- Configuration ----------------")
    for key, value in item.items():
        print(f"{key}: {value}")


--------------- Configuration ----------------
id: 1403a206-159f-42f6-bc6a-931c4fc9c63d
name: anthropic claude-3.5 sonnet_ssconfig
scenario_id: foundation-models
executable_id: aws-bedrock
parameter_bindings: [<ai_api_client_sdk.models.parameter_binding.ParameterBinding object at 0x7f8924757410>, <ai_api_client_sdk.models.parameter_binding.ParameterBinding object at 0x7f8924776cf0>]
input_artifact_bindings: []
created_at: 2025-11-13 08:23:56+00:00
scenario: None
--------------- Configuration ----------------
id: 910d7c54-736b-4dff-b19e-93faf01d68af
name: anthropic claude-3.5 sonnet_ssconfig
scenario_id: foundation-models
executable_id: aws-bedrock
parameter_bindings: [<ai_api_client_sdk.models.parameter_binding.ParameterBinding object at 0x7f8924776e70>, <ai_api_client_sdk.models.parameter_binding.ParameterBinding object at 0x7f89247753d0>]
input_artifact_bindings: []
created_at: 2025-11-13 07:25:55+00:00
scenario: None
--------------- Configuration ----------------
id: caaa9e93-6ecd-4

Check the deployment status of the embedding model(text-embedding-3-large) with its configuration

In [ ]:
response = ai_core_client.deployment.query()

# Filter items with configuration_id  
orchestration_items = [vars(rg) for rg in response.resources if getattr(rg, 'configuration_id', None) == '7c15b99e-9eea-4ae8-83ba-b1e5cf83f115']

# Print details of all matching items
for item in orchestration_items:
    print("--------------- Deployment ----------------")
    for key, value in item.items():
        print(f"{key}: {value}")


--------------- Deployment ----------------
id: d12a770eac7c91c9
configuration_id: 7c15b99e-9eea-4ae8-83ba-b1e5cf83f115
configuration_name: text-embedding-3-large-ss01
scenario_id: foundation-models
status: Status.RUNNING
target_status: TargetStatus.RUNNING
created_at: 2025-09-22 03:19:50+00:00
modified_at: 2025-11-17 03:13:08+00:00
status_message: None
status_details: None
submission_time: 2025-09-22 03:25:11+00:00
start_time: 2025-09-22 03:29:40+00:00
completion_time: None
deployment_url: https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d12a770eac7c91c9
last_operation: Operation.CREATE
latest_running_configuration_id: 7c15b99e-9eea-4ae8-83ba-b1e5cf83f115
details: {'resources': {'backend_details': {'model': {'name': 'text-embedding-3-large', 'version': 'latest'}}}, 'scaling': {'backend_details': {}}}
ttl: None


Validate if the deployment id is the same as the variable value we use

In [84]:
print (variables.EMBEDDING_DEPLOYMENT_ID)

d12a770eac7c91c9


## Help.sap.com
 

In [73]:
import os
os.environ['AICORE_RESOURCE_GROUP'] = 'DocumentGrounding'

Define the template for grounding

In [11]:
## Define the template and LLMfor grounding

from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template, TemplateValue

# Define the template
template = Template(
    messages=[
        SystemMessage("You are an expert on SAP Product features, ways to do SAP customization, and have abundant SAP development knowledge and expericence."),
        UserMessage("""Context: {{ ?grounding_response }} 
                       Anwser the question from the SAP consultant/developer with your latest knowledge to help his work:{{?query}}
                    """),
    ]
)

# Define the LLM 
llm = LLM(
    name="gpt-4o",
    parameters={
        'temperature': 0.0,
    }
)

Grounding configuration 

In [12]:

# Grounding configuration for searching SAP Help via elastic search
filters = [
    DocumentGroundingFilter(
        id="SAPHelp", 
        data_repository_type="help.sap.com"
    )
]
grounding_config = GroundingModule(
    type = GroundingType.DOCUMENT_GROUNDING_SERVICE.value,
    config = DocumentGrounding(
        input_params=["query"],
        output_param="grounding_response",
        filters=filters
    )
)
config = OrchestrationConfig(
    template = template, 
    llm = llm, 
    grounding = grounding_config
)

orchestration_service = OrchestrationService(config=config) 

In [13]:
# Run the query 
response = orchestration_service.run(
    template_values = [
        TemplateValue(
            name="query",
            value="How do I customize the WBS user status profile?",
        )
    ]
)

print(response.orchestration_result.choices[0].message.content)

Customizing the WBS user status profile in SAP involves several steps to ensure that the user statuses align with your project management requirements. Here's a detailed guide on how to customize the WBS user status profile:

1. **Access Customizing for Project System**:
   - Navigate to the SAP Customizing Implementation Guide (IMG).
   - Go to `Project System` -> `Structures` -> `Operative Structures` -> `Work Breakdown Structure (WBS)` -> `User Status Management`.

2. **Define User Status Profile**:
   - Choose `Define User Status Profile`.
   - Create a new status profile or modify an existing one. Ensure that the profile is created with the same maintenance language as in the source system if you're migrating data.

3. **Assign Object Types**:
   - Assign the relevant object types to the status profile. For WBS elements, ensure that the profile is linked to the WBS object type.
   - If migrating, ensure the technical name matches the source system.

4. **Define User Statuses**:
  

## LangChain Vector stores
 

<i>https://docs.langchain.com/oss/python/integrations/vectorstores</i>

In [3]:
import os
os.environ['AICORE_RESOURCE_GROUP'] = 'default'

<i>https://docs.langchain.com/oss/python/integrations/vectorstores/chroma</i>

### Load document
<i>https://reference.langchain.com/python/langchain_core/document_loaders/</i>


In [6]:
# Step 1: Load documents

from langchain_community.document_loaders import PyPDFDirectoryLoader
DATA_PATH = r"datafiles"
loader = PyPDFDirectoryLoader(DATA_PATH)
documents = loader.load()
print(f"Loaded {len(documents)} documents.")

Loaded 6 documents.


### Split Text 
<i>https://docs.langchain.com/oss/python/integrations/splitters</i>


In [7]:
# Step 2: Chunk documents

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=250,
    length_function=len,
)
split_documents = text_splitter.split_documents(documents)
print(f"Split into {len(split_documents)} chunks.")

Split into 54 chunks.


In [18]:
# Optional - check split result
print (split_documents)

[Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Introduction \nWe SAP are excited to announce that we have started working on a VS Code extension for \nABAP . We understand that the community has high expectations, and we want to \ncommunicate transparently about what you can expect from ABAP Development Tools for \nVS Code. In this article, we will share details about the scope of the ﬁrst release and what’s \nplanned for the future.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-

In [ ]:
[
    Document(
        metadata={
            'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'
        }, 
        page_content='Introduction \nWe SAP are excited to announce that we have started working on a VS Code extension for \nABAP . We understand that the community has high expectations, and we want to \ncommunicate transparently about what you can expect from ABAP Development Tools for \nVS Code. In this article, we will share details about the scope of the ﬁrst release and what’s \nplanned for the future.'
    ), 
    Document(
        metadata={
            'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'
        }, 
        page_content='communicate transparently about what you can expect from ABAP Development Tools for \nVS Code. In this article, we will share details about the scope of the ﬁrst release and what’s \nplanned for the future. \nSee related article: Behind the Design: How We Transformed the ABAP Development Tools \nArchitecture to Support More IDEs \nPrimary focus will be support for the ABAP Cloud Development Model'
    ), 
    Document(
        metadata={
            'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'
        }, 
        page_content='planned for the future. \nSee related article: Behind the Design: How We Transformed the ABAP Development Tools \nArchitecture to Support More IDEs \nPrimary focus will be support for the ABAP Cloud Development Model \nThe goal is to fully support all developer ﬂows related to the ABAP Cloud development \nmodel. It is not planned to support classic programming models such as Dynpro or Web \nDynpro.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Primary focus will be support for the ABAP Cloud Development Model \nThe goal is to fully support all developer ﬂows related to the ABAP Cloud development \nmodel. It is not planned to support classic programming models such as Dynpro or Web \nDynpro. \nScope of the First Release: Focus on RAP UI Services \nThe primary focus of the initial release is the development of RAP UI services. One of the'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='model. It is not planned to support classic programming models such as Dynpro or Web \nDynpro. \nScope of the First Release: Focus on RAP UI Services \nThe primary focus of the initial release is the development of RAP UI services. One of the \nbiggest requests from our users has been to bring SAP Fiori frontend development and RAP'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Dynpro. \nScope of the First Release: Focus on RAP UI Services \nThe primary focus of the initial release is the development of RAP UI services. One of the \nbiggest requests from our users has been to bring SAP Fiori frontend development and RAP \nUI service development into the same tool. We know this group of developers will beneﬁt'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='biggest requests from our users has been to bring SAP Fiori frontend development and RAP \nUI service development into the same tool. We know this group of developers will beneﬁt \nthe most when all development tasks can be performed within VS Code. That’s why this is \nthe starting point. \n \nFirst Release Targeted at Early Adopters'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='UI service development into the same tool. We know this group of developers will beneﬁt \nthe most when all development tasks can be performed within VS Code. That’s why this is \nthe starting point. \n \nFirst Release Targeted at Early Adopters \nThe initial set of object types will include everything needed to build a RAP UI service from'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='the most when all development tasks can be performed within VS Code. That’s why this is \nthe starting point. \n \nFirst Release Targeted at Early Adopters \nThe initial set of object types will include everything needed to build a RAP UI service from \nscratch, along with the basic develop/test/debug ﬂow that VS Code supports. This includes'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='First Release Targeted at Early Adopters \nThe initial set of object types will include everything needed to build a RAP UI service from \nscratch, along with the basic develop/test/debug ﬂow that VS Code supports. This includes \neditors for classes and interfaces. We plan to ship around 12+ object types. However, for'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='scratch, along with the basic develop/test/debug ﬂow that VS Code supports. This includes \neditors for classes and interfaces. We plan to ship around 12+ object types. However, for \nmany tasks outside of development ﬂows for creating RAP UI services, developers will still \nneed to switch between ABAP Development Tools for Eclipse and ABAP Development Tools \nfor VS Code.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='many tasks outside of development ﬂows for creating RAP UI services, developers will still \nneed to switch between ABAP Development Tools for Eclipse and ABAP Development Tools \nfor VS Code. \n \nABAP developer tools for VS Code will gradually catch up with ABAP developer tools \nfor Eclipse one Client Release at a Time'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='need to switch between ABAP Development Tools for Eclipse and ABAP Development Tools \nfor VS Code. \n \nABAP developer tools for VS Code will gradually catch up with ABAP developer tools \nfor Eclipse one Client Release at a Time \nIt took us 16 years to reach the current feature scope of the Eclipse plug-in. The technical'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='for VS Code. \n \nABAP developer tools for VS Code will gradually catch up with ABAP developer tools \nfor Eclipse one Client Release at a Time \nIt took us 16 years to reach the current feature scope of the Eclipse plug-in. The technical \nimplementation of our VS Code extension uses an architecture that allows us to reuse'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='for Eclipse one Client Release at a Time \nIt took us 16 years to reach the current feature scope of the Eclipse plug-in. The technical \nimplementation of our VS Code extension uses an architecture that allows us to reuse \nmuch of the existing Eclipse codebase. However, we must migrate all existing tools to the'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='implementation of our VS Code extension uses an architecture that allows us to reuse \nmuch of the existing Eclipse codebase. However, we must migrate all existing tools to the \nLanguage Server Protocol and design/build the necessary UIs on the VS Code side. While \nwe expect a much faster development timeline, this transition will not happen overnight'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='but gradually with each new client release. \n \nSame Backend Support as ABAP Development Tools for Eclipse \nSince we are reusing our existing codebase, all features available for a given release in \nEclipse can potentially be oƯered in VS Code. We plan to support all releases down to SAP \nNetWeaver 7.3 EHP1 SP04. \n \nEarly Release Means You Can Help Shape the Roadmap'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='Eclipse can potentially be oƯered in VS Code. We plan to support all releases down to SAP \nNetWeaver 7.3 EHP1 SP04. \n \nEarly Release Means You Can Help Shape the Roadmap \nJust like the Eclipse plug-in, which was not feature-complete when we started, we will \nactively engage with the ABAP developer community to help us prioritize the features you \nmiss the most. \nFile-based development ﬁrst'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='Just like the Eclipse plug-in, which was not feature-complete when we started, we will \nactively engage with the ABAP developer community to help us prioritize the features you \nmiss the most. \nFile-based development ﬁrst \nAs the name already implies Visual Studio Code is a code editor optimized for editing'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='actively engage with the ABAP developer community to help us prioritize the features you \nmiss the most. \nFile-based development ﬁrst \nAs the name already implies Visual Studio Code is a code editor optimized for editing \nsource code ﬁles. Additionally, all AI tools work best in a ﬁle-based environment. In order,'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='miss the most. \nFile-based development ﬁrst \nAs the name already implies Visual Studio Code is a code editor optimized for editing \nsource code ﬁles. Additionally, all AI tools work best in a ﬁle-based environment. In order, \nto ensure that ABAP developers get the most out of VS Code in combination with AI tools all \nobject type editing is completely ﬁle based utilizing the ABAP ﬁle formats.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='to ensure that ABAP developers get the most out of VS Code in combination with AI tools all \nobject type editing is completely ﬁle based utilizing the ABAP ﬁle formats. \n \nSAP Joule for Developers Features Coming to VS Code \nWe are planning to bring the Joule for developers features to VS Code, including predictive \ncode completion and other intelligent development capabilities.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='SAP Joule for Developers Features Coming to VS Code \nWe are planning to bring the Joule for developers features to VS Code, including predictive \ncode completion and other intelligent development capabilities. \n \nABAP Development Tools for VS Code Unlock Access to Cutting-Edge AI Tools \nOne of the reasons ABAP in VS Code is so appealing is that most cutting-edge AI'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='code completion and other intelligent development capabilities. \n \nABAP Development Tools for VS Code Unlock Access to Cutting-Edge AI Tools \nOne of the reasons ABAP in VS Code is so appealing is that most cutting-edge AI \ndevelopment innovations happen in VS Code. Thanks to the new extension, you can \nleverage the available AI VS Code extensions. Please note that ABAP development objects'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:40:21+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:40:21+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf', 'total_pages': 2, 'page': 1, 'page_label': '2'}, page_content='development innovations happen in VS Code. Thanks to the new extension, you can \nleverage the available AI VS Code extensions. Please note that ABAP development objects \nare still stored on the server side. The ﬁle system had to be implemented using the “virtual \nworkspace” technology. Not all AI tools currently support this ﬁle system, so compatibility \nmay vary.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Introduction \nCurrently, oƯicial ABAP tool support exists for SAP GUI and Eclipse. For years, users \nhave asked us to bring this support to additional IDEs. Based on the user survey \nresults from 2023 and 2025, the most requested development environment is Visual \nStudio Code (VS Code). However, many users have also expressed interest in other'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='have asked us to bring this support to additional IDEs. Based on the user survey \nresults from 2023 and 2025, the most requested development environment is Visual \nStudio Code (VS Code). However, many users have also expressed interest in other \nenvironments such as JetBrains IDEs, Neovim, or even Zed. In short, our user base \nwants more ﬂexibility when choosing their development environment.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Studio Code (VS Code). However, many users have also expressed interest in other \nenvironments such as JetBrains IDEs, Neovim, or even Zed. In short, our user base \nwants more ﬂexibility when choosing their development environment. \nSee related article: ABAP Development Tools for VS Code: Everything You Need to \nKnow \nHow Do We Transform the ABAP Development Tools Architecture to OƯer More'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='wants more ﬂexibility when choosing their development environment. \nSee related article: ABAP Development Tools for VS Code: Everything You Need to \nKnow \nHow Do We Transform the ABAP Development Tools Architecture to OƯer More \nChoice? \nWhen analyzing the challenge, we identiﬁed two major issues that needed to be \naddressed: \n \n1. The Client/Server Architecture'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Know \nHow Do We Transform the ABAP Development Tools Architecture to OƯer More \nChoice? \nWhen analyzing the challenge, we identiﬁed two major issues that needed to be \naddressed: \n \n1. The Client/Server Architecture \nABAP tools follow a client/server architecture. For each new development \nenvironment, a client layer implementation is required. This layer contains all the'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='addressed: \n \n1. The Client/Server Architecture \nABAP tools follow a client/server architecture. For each new development \nenvironment, a client layer implementation is required. This layer contains all the \ncode needed to communicate with an ABAP server, including complex wrappers for \nexisting ABAP development tools REST APIs. It ensures compatibility with system'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='environment, a client layer implementation is required. This layer contains all the \ncode needed to communicate with an ABAP server, including complex wrappers for \nexisting ABAP development tools REST APIs. It ensures compatibility with system \nreleases as far back as SAP NetWeaver 7.3 EHP1 SP04 and higher and implements'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='code needed to communicate with an ABAP server, including complex wrappers for \nexisting ABAP development tools REST APIs. It ensures compatibility with system \nreleases as far back as SAP NetWeaver 7.3 EHP1 SP04 and higher and implements \nclient-side tool logic such as the debugger, test runner, ATC, tracing, and more.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='existing ABAP development tools REST APIs. It ensures compatibility with system \nreleases as far back as SAP NetWeaver 7.3 EHP1 SP04 and higher and implements \nclient-side tool logic such as the debugger, test runner, ATC, tracing, and more. \nOverall, the client UI-independent (IDE independent) codebase consists of 2.9 million \nlines of code! \n \n2. Object Type Editors'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='client-side tool logic such as the debugger, test runner, ATC, tracing, and more. \nOverall, the client UI-independent (IDE independent) codebase consists of 2.9 million \nlines of code! \n \n2. Object Type Editors \nABAP relies heavily on object type editors. For the SAP BTP ABAP development \nenvironment alone, 88 unique editors need to be supported as of 2025. For'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='lines of code! \n \n2. Object Type Editors \nABAP relies heavily on object type editors. For the SAP BTP ABAP development \nenvironment alone, 88 unique editors need to be supported as of 2025. For \ncomparison, CAP only requires one unique editor (CDS), and Java tooling only needs \neditors for classes, interfaces and enumerations. \n \nSolution 1: Reusing Our Client Codebase as a Language Server'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='comparison, CAP only requires one unique editor (CDS), and Java tooling only needs \neditors for classes, interfaces and enumerations. \n \nSolution 1: Reusing Our Client Codebase as a Language Server \nModern development environments support language servers through the Language \nServer Protocol (LSP), which deﬁnes a standardized interface between the \ndevelopment tool and the language server.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}, page_content='(Image source) \nThis means a language server can be written in any programming language and reused \nacross multiple development environments. \nWe began exploring this approach in 2018, considering a TypeScript based language \nserver to re-implement the client component—similar to community ABAP VS Code \nextensions. While functional, this approach was not ideal for us because it required'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}, page_content='We began exploring this approach in 2018, considering a TypeScript based language \nserver to re-implement the client component—similar to community ABAP VS Code \nextensions. While functional, this approach was not ideal for us because it required \nrebuilding the entire client layer from scratch and maintaining two separate \nimplementations. This is simply not a viable solution in the long run.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}, page_content='extensions. While functional, this approach was not ideal for us because it required \nrebuilding the entire client layer from scratch and maintaining two separate \nimplementations. This is simply not a viable solution in the long run. \nFortunately, we discovered the Java VS Code extension, which provided a clever'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}, page_content='rebuilding the entire client layer from scratch and maintaining two separate \nimplementations. This is simply not a viable solution in the long run. \nFortunately, we discovered the Java VS Code extension, which provided a clever \nsolution: they "wrapped" the Eclipse Java development tools with a language server. \nThis approach allowed them to reuse the existing JDT codebase. Interestingly, most'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}, page_content='Fortunately, we discovered the Java VS Code extension, which provided a clever \nsolution: they "wrapped" the Eclipse Java development tools with a language server. \nThis approach allowed them to reuse the existing JDT codebase. Interestingly, most \nJava developers don’t realize that the VS Code Java extension is essentially Eclipse \nrunning inside VS Code!'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 1, 'page_label': '2'}, page_content='This approach allowed them to reuse the existing JDT codebase. Interestingly, most \nJava developers don’t realize that the VS Code Java extension is essentially Eclipse \nrunning inside VS Code! \nThis concept solved our problem: we can potentially reuse 2.9 million lines of our \nclient non-ui code.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}, page_content='It also means we can maintain and support Eclipse and VS Code using the same \ncodebase. Even better, users can connect to any ABAP server release supported by \nEclipse through the VS Code extension. \nSolution 2: Making Object Types Reusable via Server-Driven Development and Client-\nBased Renderers \nInitially, when building the Eclipse plug-in, we implemented new object type editors by'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}, page_content='Eclipse through the VS Code extension. \nSolution 2: Making Object Types Reusable via Server-Driven Development and Client-\nBased Renderers \nInitially, when building the Eclipse plug-in, we implemented new object type editors by \nadding persistence logic on the server in ABAP and the UI on the client side in Java. In \n2018, when the SAP BTP ABAP Environment was launched and we decided to only'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}, page_content='adding persistence logic on the server in ABAP and the UI on the client side in Java. In \n2018, when the SAP BTP ABAP Environment was launched and we decided to only \nsupport ABAP Development Tools for Eclipse for ABAP Cloud development, we had to \nadd many missing object types quickly. During this process, we realized our'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}, page_content='2018, when the SAP BTP ABAP Environment was launched and we decided to only \nsupport ABAP Development Tools for Eclipse for ABAP Cloud development, we had to \nadd many missing object types quickly. During this process, we realized our \narchitecture did not scale well. Using two programming languages was too complex \nand slow.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}, page_content='add many missing object types quickly. During this process, we realized our \narchitecture did not scale well. Using two programming languages was too complex \nand slow. \nLearning from past experiences, we looked at the server-side Dynpro and SAP Fiori \nElements programming models, which allow the implementation of ABAP-based UIs'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}, page_content='and slow. \nLearning from past experiences, we looked at the server-side Dynpro and SAP Fiori \nElements programming models, which allow the implementation of ABAP-based UIs \nusing only ABAP code and a client-side rendering engine based on UI models. \nAdopting this approach solved our scalability problem: we only need two rendering'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 2, 'page_label': '3'}, page_content='Elements programming models, which allow the implementation of ABAP-based UIs \nusing only ABAP code and a client-side rendering engine based on UI models. \nAdopting this approach solved our scalability problem: we only need two rendering \nengines for tool support—one for form-based tools and one for source-based tools.'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 3, 'page_label': '4'}, page_content='In 2020, we released editor support for number range objects—our ﬁrst server-driven \nobject type. Today, all new object types are built using our server-driven development \nframework. \nThanks to this shift, oƯering ABAP tool support in another development environment \nno longer requires 88 separate editors. It only requires two: \n\uf0b7 One editor for form-based objects'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 3, 'page_label': '4'}, page_content='framework. \nThanks to this shift, oƯering ABAP tool support in another development environment \nno longer requires 88 separate editors. It only requires two: \n\uf0b7 One editor for form-based objects \n\uf0b7 One editor for source-based objects \nConclusion \nFor years, we wanted to bring ABAP tools to more development environments, but the'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 3, 'page_label': '4'}, page_content='no longer requires 88 separate editors. It only requires two: \n\uf0b7 One editor for form-based objects \n\uf0b7 One editor for source-based objects \nConclusion \nFor years, we wanted to bring ABAP tools to more development environments, but the \nprevious architecture made this impossible. Over the past six years, we have invested'), Document(metadata={'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2025-11-18T10:42:07+08:00', 'author': 'SUN Yufeng (BD/PTD-SPR1)', 'moddate': '2025-11-18T10:42:07+08:00', 'title': 'Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx', 'source': 'datafiles/Behind the Design How We Transformed the ABAP Development Tools Architecture to Support More IDEs.pdf', 'total_pages': 4, 'page': 3, 'page_label': '4'}, page_content='\uf0b7 One editor for source-based objects \nConclusion \nFor years, we wanted to bring ABAP tools to more development environments, but the \nprevious architecture made this impossible. Over the past six years, we have invested \nsigniﬁcant eƯort into re-architecting our tools. Thanks to this work, we can now bring \nABAP tools to additional IDEs—starting with VS Code in 2026!')]

### Embedding models
<i>https://docs.langchain.com/oss/python/integrations/text_embedding</i>


In [2]:
# Step 3: Set embedding model  
 
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI, OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(deployment_id=variables.EMBEDDING_DEPLOYMENT_ID)  # Deployment ID of text-embedding-3-large

### Save to vector store
<i>https://docs.langchain.com/oss/python/integrations/vectorstores</i></br>
<i>https://reference.langchain.com/python/langchain_core/vectorstores/</i>


#### Chroma
<i>https://docs.langchain.com/oss/python/integrations/vectorstores/chroma</i>

In [8]:
# Step 4-1: Save into local Chroma DB

from langchain_chroma import Chroma
 
vector_store = Chroma.from_documents(
    documents=split_documents,
    embedding=embedding_model,
    collection_name="VS_Code",
    persist_directory="./chroma_db"
) 

#### In-memory
<i>https://docs.langchain.com/oss/python/integrations/vectorstores#in-memory</i>

In [10]:
# Step 4-2: Save into memory

from langchain_core.vectorstores import InMemoryVectorStore
 
vector_store = InMemoryVectorStore.from_documents(
    documents=split_documents,
    embedding=embedding_model,
) 

### Optional step: Query directly from Vectorstore

#### Optional: Instantiating vector_store

In [19]:
# Only run after step 4-1
# Instantiating vector_store as a Chroma instance 

from gen_ai_hub.proxy.langchain.openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

embedding_model = OpenAIEmbeddings(deployment_id=variables.EMBEDDING_DEPLOYMENT_ID)  # Embedding deployment ID 

vector_store = Chroma(
    embedding_function=embedding_model,
    collection_name="VS_Code",
    persist_directory="./chroma_db"
) 

#### Similarity search

In [12]:
# Similarity search
import json
search_results=vector_store.similarity_search(
    # "develop with AI",
    "why is ABAP in VS Code is so appealing",
    k=2)
i=1

# print results
for r in search_results:
    # print(f"Content:{r.page_content}") 
    print("————————————————————————————————————","search result",i,"—————————————————————————————————————")
    i=i+1
    print(f"Content: {r.page_content}")
    print("metadata:")
    print(json.dumps(r.metadata, indent=3))

———————————————————————————————————— search result 1 —————————————————————————————————————
Content: code completion and other intelligent development capabilities. 
 
ABAP Development Tools for VS Code Unlock Access to Cutting-Edge AI Tools 
One of the reasons ABAP in VS Code is so appealing is that most cutting-edge AI 
development innovations happen in VS Code. Thanks to the new extension, you can 
leverage the available AI VS Code extensions. Please note that ABAP development objects
metadata:
{
   "producer": "Microsoft: Print To PDF",
   "creator": "PyPDF",
   "creationdate": "2025-11-18T10:40:21+08:00",
   "author": "SUN Yufeng (BD/PTD-SPR1)",
   "moddate": "2025-11-18T10:40:21+08:00",
   "title": "Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx",
   "source": "datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf",
   "total_pages": 2,
   "page": 1,
   "page_label": "2"
}
———————————————————————————————————— search res

#### Similarity search with score

In [13]:
# Similarity search with score
import json
search_sresults = vector_store.similarity_search_with_score(
    "develop with AI",
    k=2
)

# print results
i=1
for doc,score in search_sresults:
    print("————————————————————————————————————","search result",i,"—————————————————————————————————————")
    print(f"Score: {score:3f}")
    print("Content: ")
    print(doc.page_content)
    print("metadata:")
    print(json.dumps(doc.metadata, indent=3))
    i=i+1

———————————————————————————————————— search result 1 —————————————————————————————————————
Score: 0.512821
Content: 
development innovations happen in VS Code. Thanks to the new extension, you can 
leverage the available AI VS Code extensions. Please note that ABAP development objects 
are still stored on the server side. The ﬁle system had to be implemented using the “virtual 
workspace” technology. Not all AI tools currently support this ﬁle system, so compatibility 
may vary.
metadata:
{
   "producer": "Microsoft: Print To PDF",
   "creator": "PyPDF",
   "creationdate": "2025-11-18T10:40:21+08:00",
   "author": "SUN Yufeng (BD/PTD-SPR1)",
   "moddate": "2025-11-18T10:40:21+08:00",
   "title": "Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx",
   "source": "datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf",
   "total_pages": 2,
   "page": 1,
   "page_label": "2"
}
———————————————————————————————————— search result 2 ——

#### Search by vector

In [18]:
# Search by vector
import json

from gen_ai_hub.proxy.langchain.openai import ChatOpenAI, OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(deployment_id=variables.EMBEDDING_DEPLOYMENT_ID)  # Embedding deployment ID 
 
search_results=vector_store.similarity_search_by_vector(
    embedding=embedding_model.embed_query(
        text="develop with AI"),  
    k=2)

# print results
i=1
for r in search_results:
    # print(f"Content:{r.page_content}") 
    print("————————————————————————————————————","search result",i,"—————————————————————————————————————")
    print(f"Content: {r.page_content}")
    print("metadata:")
    print(json.dumps(r.metadata, indent=3))
    i=i+1
 

———————————————————————————————————— search result 1 —————————————————————————————————————
Content: development innovations happen in VS Code. Thanks to the new extension, you can 
leverage the available AI VS Code extensions. Please note that ABAP development objects 
are still stored on the server side. The ﬁle system had to be implemented using the “virtual 
workspace” technology. Not all AI tools currently support this ﬁle system, so compatibility 
may vary.
metadata:
{
   "author": "SUN Yufeng (BD/PTD-SPR1)",
   "creator": "PyPDF",
   "moddate": "2025-11-18T10:40:21+08:00",
   "producer": "Microsoft: Print To PDF",
   "page_label": "2",
   "total_pages": 2,
   "creationdate": "2025-11-18T10:40:21+08:00",
   "page": 1,
   "source": "datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf",
   "title": "Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx"
}
———————————————————————————————————— search result 2 ———————————————————

#### Query by turning into retriever

In [20]:
# Query by turning into retriever
retriever = vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 1, "fetch_k": 5}
)
search_results=retriever.invoke("why is ABAP in VS Code is so appealing")

# print results
i=1
for r in search_results:
    # print(f"Content:{r.page_content}") 
    print("————————————————————————————————————","search result",i,"—————————————————————————————————————")
    print(f"Content: {r.page_content}")
    print("metadata:")
    print(json.dumps(r.metadata, indent=3))
    i=i+1

———————————————————————————————————— search result 1 —————————————————————————————————————
Content: code completion and other intelligent development capabilities. 
 
ABAP Development Tools for VS Code Unlock Access to Cutting-Edge AI Tools 
One of the reasons ABAP in VS Code is so appealing is that most cutting-edge AI 
development innovations happen in VS Code. Thanks to the new extension, you can 
leverage the available AI VS Code extensions. Please note that ABAP development objects
metadata:
{
   "title": "Microsoft Word - ABAP Development Tools for VS Code Everything You Need to Know .docx",
   "author": "SUN Yufeng (BD/PTD-SPR1)",
   "page_label": "2",
   "creationdate": "2025-11-18T10:40:21+08:00",
   "creator": "PyPDF",
   "source": "datafiles/ABAP Development Tools for VS Code Everything You Need to Know.pdf",
   "moddate": "2025-11-18T10:40:21+08:00",
   "page": 1,
   "producer": "Microsoft: Print To PDF",
   "total_pages": 2
}


### Usage for retrieval-augmented generation

In [ ]:
# Step 5: Instantiating vector_store as a Chroma instance 
from gen_ai_hub.proxy.langchain.openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
embedding_model = OpenAIEmbeddings(deployment_id=variables.EMBEDDING_DEPLOYMENT_ID)  # Embedding deployment ID 

vector_store = Chroma(
    embedding_function=embedding_model,
    collection_name="VS_Code",
    persist_directory="./chroma_db"
) 

In [ ]:
# Step 6: Define prompt (to avoid hallucination)
from langchain_core.prompts import PromptTemplate
template = """
You are an AI assistant. Answer the question only based on the provided context.
If the answer is not contained in the context, say "The document does not contain this information."

Context:
{context}

Question:
{question}
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)

In [ ]:
# Step 7: Initialize LLM
chat_llm = ChatOpenAI(deployment_id=variables.LLM_DEPLOYMENT_ID)  # LLM deployment ID. Here gpt-4o has been maintained
#chat_llm = ChatOpenAI(deployment_id="dac8d37a6fd75edc")  # LLM deployment ID of anthropic--claude-3.5-sonnet, tested but unsuccessful

In [ ]:
# Step 8: Define format function
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# use "|" operator to compose runnables together in a pipeline. 
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

rag_chain=(
    {
        "context":vector_store.as_retriever(search_kwargs={"k": 5}) | format_docs,  #1 Build a dictionary with context and question.
        "question":RunnablePassthrough()
    }
    | prompt                                                                        #2 Pass that dictionary to prompt (which formats it into text).
    | chat_llm                                                                      #3 Send the formatted prompt to chat_llm (the language model).
    | StrOutputParser ()                                                            #4 parse the output into a string with StrOutputParser().
)

In [ ]:
# Step 9: Define qa function
def qa(question):
    response = rag_chain.invoke(question)
    print ("Question: ",question)
    print ("Answer: ",response)
    print("———————————————————————————————————————————————————————————————————————")

In [ ]:
# Step 10: Set up questions to ask
question1="What is focus and scope as planned so far?"
question2="Why is ABAP in VS Code is so appealing?"
question3="Which difficulties have been found while checking the challenges?"
question4="What are solution?"

In [ ]:
# Step 11: Run Q&A
qa(question1)
qa(question2)
qa(question3)
qa(question4)

Question:  What is focus and scope as planned so far?
Answer:  The focus and scope as planned so far include supporting the ABAP Cloud Development Model, with a primary focus on the development of RAP UI services in the initial release. It is not planned to support classic programming models such as Dynpro or Web Dynpro.
———————————————————————————————————————————————————————————————————————
Question:  Why is ABAP in VS Code is so appealing?
Answer:  ABAP in VS Code is appealing because most cutting-edge AI development innovations happen in VS Code, and the new extension allows developers to leverage the available AI VS Code extensions.
———————————————————————————————————————————————————————————————————————
Question:  Which difficulties have been found while checking the challenges?
Answer:  The document mentions two difficulties found while checking the challenges: 

1. The architecture did not scale well.
2. Using two programming languages was too complex and slow. 

Additionally, th